# Primitive Rateless: recover the information bits x

Use the same observation editor as `y_ecc/individual_inspection.ipynb`, but decode **x**, not y. The default polynomial is $P(x)=x^5+x^2+1$, K=5, N=16. Primitivity is trusted.

Edit the channel observation, then explicitly run decoding. Initialize estimated x with independent random signs and small LLR magnitude. Every selected method gets the same initialization. Plots compare K information bits against the true x; truth is used only for diagnostics.

In [11]:
import sys
from pathlib import Path

notebooks_dir = next(
    (
        candidate
        for parent in (Path.cwd(), *Path.cwd().parents)
        for candidate in (parent, parent / "notebooks")
        if (candidate / "_shared").is_dir()
    ),
    None,
)
if notebooks_dir is None:
    raise RuntimeError("Cannot locate notebooks/_shared from this working directory")
if str(notebooks_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(notebooks_dir.resolve()))

from _shared.core.lfsr import polynomial_terms  # noqa: E402
from _shared.presentation.widgets import LLREditor  # noqa: E402
from IPython.display import display  # noqa: E402

P = polynomial_terms((0, 2, 5))  # Distinct nonzero exponents, including 0 and K.
# For a degree-32 experiment, supply your chosen degree-32 polynomial here.
K = P[-1]
N = 16  # Choose N > K if you want local parity checks and redundancy.
T = 20
SEED = None  # Use an integer to reproduce the random word and subsequent noise draws.

## Edit the observation $y'$

- Click a bit of $x$ to flip it, or **New random x** to sample a new word. Every change of $x$ resets $y'$ to the newly encoded reference. Changing $N$ also re-encodes and resets; changing $T$ preserves the observation.
- Hold the left mouse button and move over $y'$ cells. A brush acts immediately on entry and repeats every **100 ms** while held over that cell. Keyboard users can focus a cell and press Space or Enter.
- **Flip:** subtract $a s_i$ per tick. This gradually crosses zero; it is not instantaneous sign negation.
- **Suppress:** multiply by $10^{-d/20}$ per tick. Here dB is explicitly an **amplitude attenuation convention for the LLR**, not an SNR conversion. Infinite dB erases immediately.
- **Restore:** add $a s_i$ per tick. It can increase confidence above one. **Reset** instead sets all cells exactly to $s$.
- **Add noise:** add independent $\mathcal N(0,\sigma^2)$ samples to the current $y'$. Repeated clicks accumulate noise. The global control sets $\sigma$ directly in LLR units (default: 0.5 LLR, step: 0.1, minimum: 0).

Green means the sign agrees with the reference; red means it disagrees. Zero is white. Color strength is $1-\exp(-|L|/2)$, on a fixed scale shared with the evolution plot. Hover for the full numeric value.

There is no fixed upper cap on $N$ or $T$. Large words still require rendering and transmitting $N$ editable cells, so the browser can become the bottleneck. Changing $T$ only sets the next explicit run length; it never starts a computation.

In [12]:
# Recreating the editor resets the experiment input; it does not decode.
if "editor" in globals():
    globals()["editor"].close()
import importlib

LLREditor = importlib.reload(sys.modules[LLREditor.__module__]).LLREditor
editor = LLREditor(n=N, iterations=T, terms=P, seed=SEED)
display(editor)

## Messages through the generator graph

For the convention $y=xG$, G has shape K by N. Column j defines
$y_j=\bigoplus_{i\in S_j}x_i$, where $S_j=\{i:G_{ij}=1\}$.

The channel observation $\ell_j$ is fixed throughout decoding. Initialize $L_i^{(0)}$ to random signs of magnitude `INIT_LLR_SCALE`. One synchronous iteration is

$$m_{j\to i}^{(t)}=\operatorname{softxor}\left(\ell_j,\{L_r^{(t)}:r\in S_j\setminus\{i\}\}\right),$$
$$L_i^{(t+1)}=L_i^{(t)}+\sum_{j:i\in S_j}m_{j\to i}^{(t)}.$$

The observation participates in every column message: without it, messages would not depend on the received word. All messages use the previous iteration. A singleton column supplies a direct channel opinion; an unconnected x bit retains its initial LLR. There are no shifted polynomial checks in this decoder. The polynomial is used only to construct the encoding graph.

Sparse column supports are generated by an LFSR with arbitrary-width integer masks; no dense G or companion matrix is allocated. Storage still scales with nnz(G), which can approach K*N. Iterations are compiled with Numba.

The available soft-XOR variants are exact tanh/box-plus, normalized min-sum (coefficient once per message), and sqrt-sign (sign times geometric mean of all message inputs, including the channel). This additive update reuses information, rather than passing extrinsic BP messages; confidence can grow even for wrong signs.

## Decode the current observation: head-to-head

`INIT_LLR_SCALE` controls the magnitude of random initial x beliefs. `INIT_SEED` reproduces the signs independently of the editor's random word. Both are shared across the selected decoders. Changing y' does not run this cell or overwrite previous results.

The decoder accepts observed `(B,N)` LLRs and returns final `(B,K)` LLRs or `(B,T+1,K)` histories with `return_history=True`. `x_initial` can supply explicit initial beliefs; otherwise the decoder draws random signs using `init_scale` and `seed`. No decoder receives the true x.

In [13]:
import importlib
from functools import partial
from time import perf_counter

import _shared.application.comparison as comparison
import _shared.core.decoders._adt as softxor
import numpy as np
from _shared.core.decoders.x import avail_softmajvote

# Reload implementations without recreating the editor or changing its observation.
softxor = importlib.reload(softxor)
avail_softmajvote = importlib.reload(avail_softmajvote)
comparison = importlib.reload(comparison)

DECODERS = {
    "avail-softmajvote(tanh)": partial(
        avail_softmajvote.decode, softxor=softxor.Tanh()
    ),
    "avail-softmajvote(sqrt-sign)": partial(
        avail_softmajvote.decode, softxor=softxor.SqrtSign()
    ),
    "avail-softmajvote(min-sum(0.8))": partial(
        avail_softmajvote.decode,
        softxor=softxor.NormalizedMinSum(coefficient=0.8),
    ),
}

snapshot = editor.snapshot()
true = snapshot["information"]
terms = snapshot["terms"]
INIT_LLR_SCALE = 0.1
INIT_SEED = 0
if not np.isfinite(INIT_LLR_SCALE) or INIT_LLR_SCALE <= 0:
    raise ValueError("INIT_LLR_SCALE must be finite and positive")
x_initial = (
    1.0 - 2 * np.random.default_rng(INIT_SEED).integers(0, 2, len(true))
) * INIT_LLR_SCALE
started = perf_counter()
histories = comparison.run_x_decoders(
    DECODERS,
    snapshot["initial"],
    x_initial,
    terms=terms,
    iterations=snapshot["iterations"],
)
print(
    f"Compared {len(histories)} decoders: N={len(snapshot['initial'])}, K={terms[-1]}, "
    f"T={snapshot['iterations']} in {perf_counter() - started:.3f} s"
)

Compared 3 decoders: N=16, K=5, T=20 in 3.994 s


## Iteration versus metrics for x

Compare sign error rate (zero counts as half an error), erasure fraction, mean logistic loss, and mean signed margin against the **information bits x**. Recurrence syndrome is deliberately omitted: x is not a length-N recurrence codeword.

Decoder selectors and metric selectors combine independently. All plots use completed histories; switching selections never runs decoding.

In [14]:
import importlib

import _shared.application.metrics as diagnostics
import _shared.presentation.plot_controls as plot_controls
import _shared.presentation.plots as plots

diagnostics = importlib.reload(diagnostics)
plot_controls = importlib.reload(plot_controls)
plots = importlib.reload(plots)
metrics_view = plots.compare(
    histories,
    true,
    kind="metrics",
    terms=None,
    previous=globals().get("metrics_view"),
)
display(metrics_view)

ComparisonView(children=(FigureWidget({
    'data': [{'hovertemplate': 'Iteration %{x}<br>%{y:.6g}<extra>%{ful…

## Iteration versus information bits

There are K curves per decoder, with $q_i^{(t)}=(1-2x_i)L_i^{(t)}$. Positive is correct, negative is wrong, zero is erased. Select decoders and bits independently in the legend.

In [15]:
import importlib

import _shared.presentation.plot_controls as plot_controls
import _shared.presentation.plots as plots

plot_controls = importlib.reload(plot_controls)
plots = importlib.reload(plots)
bits_view = plots.compare(
    histories,
    true,
    kind="bits",
    previous=globals().get("bits_view"),
)
display(bits_view)

ComparisonView(children=(FigureWidget({
    'data': [{'hovertemplate': 'Iteration %{x}<br>%{y:.6g}<extra>%{ful…

## Evolution of the estimated x LLRs

Each row contains K estimated information-bit LLRs, starting with the shared random initialization at t=0. Green/red indicate agreement/disagreement with true x; magnitude controls color strength. Hover displays the raw LLR. Select a decoder to inspect its trajectory.

In [16]:
import importlib

import _shared.presentation.plot_controls as plot_controls
import _shared.presentation.plots as plots

plot_controls = importlib.reload(plot_controls)
plots = importlib.reload(plots)
evolution_view = plots.compare(
    histories,
    true,
    kind="evolution",
    previous=globals().get("evolution_view"),
)
display(evolution_view)

ComparisonView(children=(FigureWidget({
    'data': [{'colorbar': {'len': 0.92,
                           'ou…